# One school campus, building by building: University High School, Parkville

What does one shape per school hide? One school drawn as its buildings, rolled up to the site, with estimated floor
space per student.

The other school showcases draw one shape per school and divide its grounds (the land it sits on, buildings included)
by its students. That is a site-level figure. This notebook takes one campus and reports it building by building, then
rolls the buildings back up to the site, so the two can be compared. Students are February FTE enrolments (Victorian
Department of Education), not a headcount.

The shapes in `data/geometry/` were made in [Plotlay](https://www.getplotlay.com): the site from the City of Melbourne's
two property parcels (University High School and the Elizabeth Blackburn School of Sciences), and each building picked
from the council's 2023 building footprints, the parts of one structure joined by `structure_id`. The site's roll-up
(how many buildings, their footprint and floor area) was worked out in Plotlay from the buildings that stand inside it.

It reads only the files in `../data`. Enrolments are the school's: they are divided at site level only, never split
across buildings. Floor space is an estimate from the council's heights, said wherever it is used.

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*colormapping.*')
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

DATA = '../data'
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
INK, MUTED = '#1d4f91', '#5b6573'
ERA = {'1930, built (CLUE)': '#E69F00', '2014, built (CLUE)': '#56B4E9', 'On the 2014 outlines': '#009E73', 'Shown from the 2018 footprints': '#F0E442', 'Shown from the 2023 footprints': '#0072B2'}
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})

## Sources and licence

Building outlines 2014 and 2015, building footprints 2018, 2020 and 2023, property boundaries and the Census of Land Use
and Employment (CLUE) building information: City of Melbourne Open Data, CC BY 4.0. February FTE enrolments: Victorian
Department of Education, All Schools FTE Enrolments (DataVic), CC BY 4.0. Nothing comes from OpenStreetMap or aerial
imagery. Full credits are in `NOTICE`. `data/sample/` is sample data and is used only in the last section, labelled so.

## Load

In [ ]:
b = pd.read_csv(f'{DATA}/buildings/buildings.csv', dtype={'structure_id': str})
fy = pd.read_csv(f'{DATA}/buildings/footprint_by_year.csv')
enr = pd.read_csv(f'{DATA}/site/enrolments.csv')
clue = pd.read_csv(f'{DATA}/site/clue_records.csv')
roll = pd.read_csv(f'{DATA}/site/site_rollup.csv')
shapes = gpd.read_file(f'{DATA}/geometry/campus_buildings.geojson').to_crs(7855)
site = gpd.read_file(f'{DATA}/geometry/campus_site.geojson').to_crs(7855)
assert b.key.is_unique and set(b.key) == set(shapes.key)
print(len(b), 'buildings;', len(site), 'site shape in', int(roll.parts[0]), 'parts;', enr.year.min(), 'to', enr.year.max(), 'enrolments')
b[['key', 'structure_id', 'footprint_m2', 'height_m', 'roof', 'first_mapped', 'era', 'storeys_est', 'storeys_from', 'floor_m2_est', 'clue_construction_year', 'clue_floors', 'clue_accessibility_rating']]

## The site as one shape

What a one-shape-per-school map knows: the site's area and the school's enrolment.

In [ ]:
site_m2 = site.geometry.area.sum()
fte = enr.set_index('year').fte_students
print(f'Site: {site_m2:,.0f} m² (Plotlay worked out {roll.area_m2[0]:,.0f} m²)')
for y in (fte.index.min(), 2025):
    print(f'{y}: {fte[y]:,.1f} students, {site_m2 / fte[y]:.1f} m² of grounds each')
print(f'Enrolment {fte.index.min()} to {fte.index.max()}: {fte.iloc[-1] / fte.iloc[0] - 1:+.0%}')

The council's land use census (CLUE) has one record for the school's property in every census year, and one for the
science school since it opened. It describes a building, not the campus:

In [ ]:
clue.groupby('clue_property_id').agg(first=('census_year', 'min'), last=('census_year', 'max'), years=('census_year', 'nunique'), name=('building_name', 'last'), built=('construction_year', 'last'), floors=('floors', 'last'), access=('accessibility_rating', 'last'))

## Building by building

The council's 2023 footprints show the buildings on the site. `height_m` is the top of a building's highest part above
the lowest base of the structure, as the council measures it (on a sloping site the uphill side reads a little taller).
Storeys and floor area are estimated: height divided by 4 m floor to floor, 2 m taken off a pitched roof, never more than
the CLUE floors where the census has a record for that building.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
site.boundary.plot(ax=ax, color=INK, lw=1.5, ls='--')
sh = shapes.merge(b[['key', 'era']], on='key')
for era, g in sh.groupby('era'): g.plot(ax=ax, color=ERA[era], edgecolor=INK, lw=.6, label=era)
for _, r in sh.iterrows():
    if r.geometry.area > 150: ax.annotate(r.key, r.geometry.representative_point().coords[0], ha='center', fontsize=8)
ax.set_axis_off(); ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc=ERA[e]) for e in ERA if e in set(sh.era)], labels=[e for e in ERA if e in set(sh.era)], fontsize=8, loc='lower right')
ax.set_title('Buildings by era: the CLUE year built, else the first council layer that shows them'); plt.show()

In [ ]:
fp_total, floor_total = b.footprint_m2.sum(), b.floor_m2_est.sum()
assert abs(fp_total - roll.footprint_m2[0]) < 2 and abs(floor_total - roll.floor_m2_est[0]) < 1   # Plotlay's roll-up
print(f'{len(b)} buildings: footprint {fp_total:,.0f} m² ({fp_total / site_m2:.0%} of the site), floor area {floor_total:,.0f} m² (estimated)')
print(f'Floor over footprint: {floor_total / fp_total:.2f}; floor over grounds: {floor_total / site_m2:.2f}')
share = (b.set_index('key').floor_m2_est / floor_total).sort_values(ascending=False)
print('Share of the floor:', ', '.join(f'{k} {v:.0%}' for k, v in share.head(4).items()))

## Grounds are not floor space (site level, 2025)

In [ ]:
y = 2025
per = pd.Series({'grounds (the site)': site_m2, 'footprints': fp_total, 'floor space (estimated)': floor_total}) / fte[y]
ax = per.plot.barh(color=[INK, '#7fb0d8', '#E69F00'], figsize=(8, 2.8)); ax.invert_yaxis()
ax.set_xlabel(f'm² per student, {y}')
for i, v in enumerate(per): ax.text(v + .2, i, f'{v:.1f}', va='center')
plt.show(); per.round(1)

## When the buildings appeared

Each building's footprint on the council's snapshots (blank: not on that layer). The six single-storey buildings of
about 200 m² are on the 2023 footprints (captured January 2022) and not on the 2020 ones.

In [ ]:
wide = fy.pivot(index='key', columns='year', values='footprint_m2')
print(wide.notna().sum().rename('buildings shown'))
new = wide[wide[2020].isna() & wide[2023].notna()]
print(f'New between the 2020 and 2023 footprints: {len(new)} buildings, {new[2023].sum():,.0f} m²; enrolment {fte[2020]:,.0f} in 2020, {fte[2025]:,.0f} in 2025')
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax[0].plot(fte.index, fte.values, color=INK, marker='o'); ax[0].set_ylabel('students (FTE)'); ax[0].set_ylim(0, None)
ax[1].bar(wide.columns, wide.sum(), color='#E69F00'); ax[1].set_ylabel('footprint m²')
plt.show()

## Accessibility, where it is recorded

CLUE rates the accessibility of the buildings it records (0 not rated to 3 high). On this campus that is two of the
thirteen buildings; the rest have no record at all, which is itself the finding: a site-level record cannot say which
building a wheelchair user can get into.

In [ ]:
b[['key', 'clue_building_name', 'clue_accessibility_rating', 'clue_accessibility_type']].astype(object).fillna('no CLUE record')

## Sample data: facilities on the same buildings

Everything from here on reads `data/sample/`, which is sample data: the figures are not about University High School
or any real building. They show what a facilities team links once each building is a keyed shape: the asset register,
risk, work orders, planned maintenance, essential safety measures, the asbestos register, meters and the capital plan,
in the shape a CAFM or CMMS exports them. `data/sample/README.md` is the data dictionary and says how each column was
drawn. Nothing above uses these tables.

In [ ]:
S = f'{DATA}/sample'
assets = pd.read_csv(f'{S}/assets.csv'); risk = pd.read_csv(f'{S}/asset_risk.csv')
wo = pd.read_csv(f'{S}/work_orders.csv', parse_dates=['raised', 'target', 'completed'])
pm = pd.read_csv(f'{S}/pm_compliance.csv'); esm = pd.read_csv(f'{S}/esm_register.csv'); asb = pd.read_csv(f'{S}/asbestos_register.csv')
cap = pd.read_csv(f'{S}/capital_works.csv'); summary = pd.read_csv(f'{S}/building_summary.csv'); space = pd.read_csv(f'{S}/space_use.csv')
for t in (assets, risk, wo, pm, esm, asb, cap, summary, space): assert (t.data_status == 'Sample data').all()
assert set(summary.key) == set(b.key) and set(assets.building.dropna()) <= set(b.key)
ar = assets.merge(risk[['asset_id', 'likelihood_1to5', 'consequence_1to5', 'risk_score', 'risk_rating', 'risk_drivers']], on='asset_id')
open_wo = wo[~wo.status.isin(['Completed', 'Cancelled, missed'])]
print(f"Sample data: {len(assets)} assets, {(risk.risk_score >= 12).sum()} at high or extreme risk; {len(open_wo)} work orders open, {(wo.overdue == 'Yes').sum()} overdue;",
      f"planned maintenance on time {100 * pm.on_time.sum() / pm.scheduled.sum():.1f}%; essential safety measures overdue {(esm.status == 'Overdue').sum()} of {len(esm)}; backlog A${cap.estimate_aud.sum():,.0f}")

### The register inside the buildings

Each asset is a point. A spatial join puts it in the building whose outline holds it: the same roll-up Plotlay does
from the assets layer into the buildings layer. The two hydrants in the grounds fall in no building.

In [ ]:
pts = gpd.GeoDataFrame(ar, geometry=gpd.points_from_xy(ar.lng, ar.lat), crs=4326).to_crs(7855)
joined = gpd.sjoin(pts, shapes[['key', 'geometry']], how='left', predicate='within')
per = joined.groupby('key').agg(assets=('asset_id', 'count'), high_or_extreme=('risk_score', lambda s: (s >= 12).sum()), highest=('risk_score', 'max'))
check = per.merge(summary.set_index('key')[['assets', 'risk_high_or_extreme']].add_suffix('_table'), left_index=True, right_index=True)
assert (check.assets == check.assets_table).all() and (check.high_or_extreme == check.risk_high_or_extreme).all()
print('In the grounds:', joined.key.isna().sum(), 'assets')
fig, ax = plt.subplots(figsize=(10, 5.5))
sh = shapes.merge(summary[['key', 'risk_high_or_extreme']], on='key')
sh.plot(ax=ax, column='risk_high_or_extreme', cmap='Oranges', edgecolor=INK, lw=.6, legend=True, legend_kwds={'label': 'Assets at high or extreme risk (Sample data)', 'shrink': .6})
col = {'Low': '#cfd8e3', 'Medium': '#f3c46b', 'High': '#e07a3a', 'Extreme': '#a32a17'}
for r, g in pts.groupby('risk_rating'): g.plot(ax=ax, color=col[r], markersize={'Low': 4, 'Medium': 7, 'High': 12, 'Extreme': 30}[r], edgecolor=INK, lw=.3, label=r)
ax.legend(title='Asset risk (Sample data)', loc='lower right', fontsize=8); ax.set_axis_off(); ax.set_title('Sample data: where the risk sits'); plt.show()
per

### Risk: likelihood times consequence

In [ ]:
matrix = risk.pivot_table(index='likelihood_1to5', columns='consequence_1to5', values='asset_id', aggfunc='count', fill_value=0).sort_index(ascending=False)
print(risk.risk_rating.value_counts().reindex(['Low', 'Medium', 'High', 'Extreme']))
ar.sort_values(['risk_score', 'consequence_1to5'], ascending=False).head(10)[['asset_id', 'asset_type', 'building', 'risk_score', 'risk_rating', 'risk_drivers']]

In [ ]:
matrix

### Work orders

Open now by priority and how long they have been open, and raised each month, reactive against planned.

In [ ]:
bins = pd.cut(open_wo.age_days, [-1, 7, 30, 90, 10_000], labels=['up to a week', '8 to 30 days', '31 to 90 days', 'over 90 days'])
print(pd.crosstab(open_wo.priority, bins, margins=True))
monthly = wo.groupby([wo.raised.dt.to_period('M'), 'wo_type']).size().unstack(fill_value=0)
ax = monthly.plot.bar(figsize=(10, 3.5), color={'Reactive': '#e0884c', 'Planned': '#8fb3d6'}); ax.set_xlabel(''); ax.set_ylabel('work orders raised'); ax.set_title('Sample data: raised each month'); plt.show()

### Planned maintenance and essential safety measures

In [ ]:
pmm = pm.groupby('month')[['scheduled', 'on_time']].sum(); pmm['on_time_pct'] = (100 * pmm.on_time / pmm.scheduled).round(1)
print(pmm.on_time_pct.to_string())
pd.crosstab(esm.building, esm.status)

### Meters from the BMS trend log

Hourly kWh per building meter (point names such as `B02-EM-01.kWh`), four weeks of term. The share used outside
8 am to 4 pm on school days, and B02's week as a heat map: it does not switch off at night.

In [ ]:
tr = pd.read_csv(f'{S}/meter_trends.csv', parse_dates=['timestamp'])
e = tr[tr.unit == 'kWh'].assign(key=lambda d: d.point.str[:3], wd=lambda d: d.timestamp.dt.weekday, hr=lambda d: d.timestamp.dt.hour)
day = (e.wd < 5) & (e.hr >= 8) & (e.hr < 16)
after = (e[~day].groupby('key').value.sum() / e.groupby('key').value.sum() * 100).round(1)
assert (after - summary.set_index('key').after_hours_share_pct.dropna()).abs().max() < 0.11
heat = e[e.key == 'B02'].pivot_table(index='wd', columns='hr', values='value', aggfunc='mean')
fig, ax = plt.subplots(figsize=(10, 3)); im = ax.imshow(heat, aspect='auto', cmap='Oranges'); ax.set_yticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
ax.set_xlabel('hour'); ax.set_title('Sample data: B02 mean kWh by hour and day'); plt.colorbar(im, ax=ax, shrink=.8); plt.show()
after.sort_values(ascending=False)

### The capital plan and the asbestos register

In [ ]:
print(asb.groupby(['building', 'risk']).size().unstack(fill_value=0))
cap.pivot_table(index='building', columns='priority', values='estimate_aud', aggfunc='sum', fill_value=0, margins=True)